In [3]:
# pip install pyarrow numpy
import pyarrow.parquet as pq
import numpy as np

# Path to the Parquet file (or a directory of files)
parquet_path = "../data/raw/nng_15x15.parquet"

# Open the file as a ParquetFile object – this does NOT load data yet
pf = pq.ParquetFile(parquet_path)

# Choose the column you want statistics for
col_name = "requires_search"

# Accumulators for streaming formulas
count = 0
sum_ = 0.0
sum_sq = 0.0
min_ = np.inf
max_ = -np.inf

# Iterate over row‑groups (or batches) to keep memory bounded
for rg_index in range(pf.num_row_groups):
    # Read only the target column of the current row‑group
    table = pf.read_row_group(rg_index, columns=[col_name])
    arr = table[col_name].to_numpy()          # NumPy array, still in memory for this batch

    n = arr.size
    count += n
    sum_ += arr.sum()
    sum_sq += (arr ** 2).sum()
    min_ = min(min_, arr.min())
    max_ = max(max_, arr.max())

# Final statistics
mean = sum_ / count
variance = (sum_sq - (sum_ ** 2) / count) / (count - 1)
std_dev = np.sqrt(variance)

print(f"Count   : {count}")
print(f"Mean    : {mean:.6f}")
print(f"Std Dev : {std_dev:.6f}")
print(f"Min     : {min_}")
print(f"Max     : {max_}")

Count   : 100000
Mean    : 0.053770
Std Dev : 0.225564
Min     : False
Max     : True


In [6]:
# Print the first parquet row group to verify the data
first_row_group = pf.read_row_group(0)
print("\nFirst row group:")
print(first_row_group)


First row group:
pyarrow.Table
puzzle_id: int64
row_clues: list<element: list<element: int16>>
  child 0, element: list<element: int16>
      child 0, element: int16
col_clues: list<element: list<element: int16>>
  child 0, element: list<element: int16>
      child 0, element: int16
solution: list<element: list<element: int8>>
  child 0, element: list<element: int8>
      child 0, element: int8
intermediate_solutions: list<element: list<element: list<element: double>>>
  child 0, element: list<element: list<element: double>>
      child 0, element: list<element: double>
          child 0, element: double
intermediate_methods: list<element: string>
  child 0, element: string
grid_density: double
grid_height: int32
grid_width: int32
steps: int32
requires_search: bool
one_step_rounding: bool
----
puzzle_id: [[0,1,2,3,4,...,995,996,997,998,999]]
row_clues: [[[[2,2,1,1],[6,1,1,1],...,[2,2,1,1,1,1],[2,1,1,2]],[[3,2,1,1],[6,4,1],...,[1,1,1,3],[2,1,8]],...,[[1,3,3,1,1],[2,2,3,2,1],...,[2,1,1,